In [ ]:
from pathlib import Path
from mutools.plotting import run, prism_schematic, use_style
import mutools.plotting as mp

# Configure once — applies everywhere
mp.saver.configure(fmt="pdf", dpi=300, rasterized=True, ytick_precision=2)

use_style("rootlike")
INPUT = Path('/Users/mueller/Projects/spineosc')
OUTPUT = INPUT / 'detsys/plotting/coverage_figures'
# Staged first; copy into /Users/mueller/Projects/spineosc_technote/figures/data_sim_comparisons once checked.
TECHNOTE = INPUT / 'docs' / 'data_sim_comparisons_v1.1.0'
RUNS = INPUT / 'runs' / 'cov_v2'      # PROfit outputs of xml/coverage (scripts/dent_study/run_cov_v2.py)

In [ ]:
# Data/MC for every detector-model coverage variable. Each entry maps the
# PROfit tag stem to its axis label, the per-channel fit range (which must
# match the <bins> min/max of that variable's xml/coverage/<stem>.xml), and
# the detector layout of that config.
#
# The kinematic/TKI block is in NATIVE units (MeV, MeV/c, rad), not the GeV
# convention of xml/varset1_devsample.xml, matching the detvar ratio axes.
#
# Position variables are SINGLE-DETECTOR configs: a coordinate only means
# something in its own geometry, so SBND and ICARUS each get their own file
# and detector indices restart at 0 inside it. ICARUS x is folded (|x|), as
# the cryostats leave no TPC in |x| < 70 cm.

# Detector layouts: (detector labels, data POT watermark per detector index).
# One [[plot]] block is emitted per detector so each panel gets its own POT.
ALL    = (['SBND', 'ICARUS Run 2', 'ICARUS Run 4'], ['4.51e18 POT', '1.97e19 POT', '1.89e19 POT'])
SBND   = (['SBND'],                                 ['4.51e18 POT'])
ICARUS = (['ICARUS Run 2', 'ICARUS Run 4'],         ['1.97e19 POT', '1.89e19 POT'])

VARIABLES = {
    # stem:                 (xlabel,                       ch0 xlim,          ch1 xlim,          layout)
    'muon_score':           ('Muon PID Score',             [0.30, 1.00],      [0.30, 1.00],      ALL),
    'proton_score':         ('Proton PID Score',           [0.40, 1.00],      [0.40, 1.00],      ALL),
    'muon_primary_score':   ('Muon Primary Score',         [0.80, 1.00],      [0.80, 1.00],      ALL),
    'proton_primary_score': ('Proton Primary Score',       [0.10, 1.00],      [0.10, 1.00],      ALL),
    # Calo/MCS-CSDA bias variables are left out of this run.
    #'muon_calo_csda':      ('Muon Calo-CSDA Bias',        [-0.80, 0.20],     [-0.80, 0.20],     ALL),
    #'proton_calo_csda':    ('Proton Calo-CSDA Bias',      [-0.90, 0.00],     [-0.90, 0.00],     ALL),
    #'muon_mcs_csda':       ('Muon MCS-CSDA Bias',         [-0.50, 1.00],     [-0.50, 1.00],     ALL),
    'proton_ke':            ('Proton KE [MeV]',            [40.0, 600.0],     [40.0, 600.0],     ALL),
    'dp':                   (r'$\delta p$ [MeV/c]',        [0.0, 1000.0],     [0.0, 1000.0],     ALL),
    'dpt':                  (r'$\delta p_{T}$ [MeV/c]',    [0.0, 1000.0],     [0.0, 1000.0],     ALL),
    'dalphat':              (r'$\delta \alpha_{T}$ [rad]', [0.0, 3.14],       [0.0, 3.14],       ALL),
    'dphit':                (r'$\delta \phi_{T}$ [rad]',   [0.0, 3.14],       [0.0, 3.14],       ALL),
    'muon_azimuth':         ('Muon Azimuthal Angle [rad]', [-3.14, 3.14],     [-3.14, 3.14],     ALL),
    'vertexx_sbnd':         ('Vertex X [cm]',              [-200.0, 200.0],   [-200.0, 200.0],   SBND),
    'vertexy_sbnd':         ('Vertex Y [cm]',              [-200.0, 200.0],   [-200.0, 200.0],   SBND),
    'vertexz_sbnd':         ('Vertex Z [cm]',              [0.0, 450.0],      [0.0, 450.0],      SBND),
    'muon_endx_sbnd':       ('Muon End X [cm]',            [-200.0, 200.0],   [-200.0, 200.0],   SBND),
    'muon_endy_sbnd':       ('Muon End Y [cm]',            [-200.0, 200.0],   [-200.0, 200.0],   SBND),
    'muon_endz_sbnd':       ('Muon End Z [cm]',            [0.0, 500.0],      [0.0, 500.0],      SBND),
    'vertexx_icarus':       ('|Vertex X| [cm]',            [70.0, 350.0],     [70.0, 350.0],     ICARUS),
    'vertexy_icarus':       ('Vertex Y [cm]',              [-185.0, 125.0],   [-185.0, 125.0],   ICARUS),
    'vertexz_icarus':       ('Vertex Z [cm]',              [-900.0, 900.0],   [-900.0, 900.0],   ICARUS),
    'muon_endx_icarus':     ('|Muon End X| [cm]',          [70.0, 350.0],     [70.0, 350.0],     ICARUS),
    'muon_endy_icarus':     ('Muon End Y [cm]',            [-185.0, 125.0],   [-185.0, 125.0],   ICARUS),
    'muon_endz_icarus':     ('Muon End Z [cm]',            [-900.0, 900.0],   [-900.0, 900.0],   ICARUS),
}

# Tag shared by the PROfit runs that produced the histogram files.
TAG = 'test09142026'

# Add PROfit's data/MC chi2 (from the file's "chi2" tree) to each legend. The
# coverage runs are shape-only (area-normalized), so it is labelled as such.
SHOW_CHI2 = True
CHI2_LABEL = r'$\chi^2_{shape}$'

# Two legend columns halve the legend's height, so the padded ylim below needs
# far less headroom to keep the legend clear of the data.
LEGEND_NCOL = 2

# TECHNOTE file naming: datamc_<detector/run>_<variable>_<singlep/multip>.pdf
DETECTOR_TOKENS = {'SBND': 'sbnd_run1', 'ICARUS Run 2': 'icarus_run2', 'ICARUS Run 4': 'icarus_run4'}
CHANNEL_TOKENS = ['singlep', 'multip']

mp.saver.configure(mark_hadj=0.055)

import math
import shutil
import numpy as np
import matplotlib.pyplot as plt
from mutools.plotting.profit import ProfitPlotData, TraceType


def make_config(input_file, output, stem, xlabel, xlim, ylim, labels, pot, channel, detector, legend_loc,
                variable=0, show_chi2=None, ylabel='Area Normalized'):
    """TOML for ONE histogram. output=None renders without saving.

    `variable` selects which variable of the file to draw. PROfit writes the
    chi2 tree for its fitting variable only, so show_chi2 defaults to
    SHOW_CHI2 for var0 and to False for any other variable."""
    if show_chi2 is None:
        show_chi2 = SHOW_CHI2 and variable == 0
    ylim_line = f"ylim = {list(ylim)}" if ylim is not None else ""
    return rf"""
[general]
input = '{str(input_file)}'
output = '{str(output) if output is not None else ""}'
savefig = {str(output is not None).lower()}
code_version = 'v3.0.4'
selection_version = 'v1.1.0'
subchannels = ['Target $\nu_\mu$ CC', 'Other $\nu_\mu$ CC', '$\nu_e$ CC', '$\nu$ NC', 'Out-of-time Cosmics', 'In-time Cosmics']
channels = ['Single-Proton', 'Multi-Proton']
detectors = {labels}
counter_index = 4

[[plot]]
type = 'histogram'
variable = {variable}
channel = {channel}
detectors = [{detector}]
colors = ['C0', 'C1', 'C2', 'C4', 'C3', 'C5']
xlabel = '{xlabel}'
ylabel = '{ylabel}'
xlim = {xlim}
{ylim_line}
rlim = [0.0, 2.0]
ratio = 'data'
watermark = '$\bf{{SBN}}$ Analysis-in-Progress'
disable_systematics = true
counter_fmt = '.1%'
right_watermark = '{pot[detector]}'
band_label = "Total Uncertainty"
show_chi2 = {str(show_chi2).lower()}
chi2_label = '{CHI2_LABEL}'
legend_ncol = {LEGEND_NCOL}
legend_loc = '{legend_loc}'
"""


def content_tops(input_file, channel, detector, variable=0):
    """Per-bin (low edge, high edge, top) of everything on the main panel:
    MC stack, total-uncertainty band, and data +1 sigma."""
    d = ProfitPlotData(input_file)
    cv = d.get_trace(f"{variable}:0:{detector}:{channel}:total:CV", TraceType.HIST_CONTENTS)
    band = d.get_trace(f"{variable}:0:{detector}:{channel}:total:CV", TraceType.HIST_ERROR_BAND)
    dat = d.get_trace(f"{variable}:0:{detector}:{channel}:total:DATA", TraceType.HIST_CONTENTS)
    derr = dat[:, 4] if dat.shape[1] > 4 else np.sqrt(np.maximum(dat[:, 3], 0.0))
    top = np.nanmax(np.vstack([cv[:, 3], band[:, 1] + band[:, 3], dat[:, 3] + derr]), axis=0)
    return cv[:, 1], cv[:, 2], top


def nice_ceil(x):
    """Round up to two significant figures."""
    e = 10 ** math.floor(math.log10(x))
    return float(f"{math.ceil(round(x / e * 10, 6)) / 10 * e:.2g}")


def padded_ylim(input_file, stem, xlabel, xlim, labels, pot, channel, detector,
                locs=('upper right', 'upper left'), margin=0.95, max_iter=6, variable=0):
    """Smallest (2 s.f.) y-axis maximum at which no drawn content reaches into
    the legend, and the legend corner that achieves it.

    The legend is pinned to a top corner (matplotlib's 'best' placement slides
    a short legend down as ymax grows, which never converges). For each corner
    a pass renders, reads the legend's bottom edge as an axes fraction, and
    solves for the ymax that puts the content beneath it at `margin` of that
    edge; a second pass confirms, since tick-label widths can nudge the legend.
    The corner needing less headroom wins."""
    lo, hi, top = content_tops(input_file, channel, detector, variable)
    inx = (hi > xlim[0]) & (lo < xlim[1])
    floor = nice_ceil(1.1 * top[inx].max())
    best = None
    for loc in locs:
        ymax = floor
        for _ in range(max_iter):
            plt.close('all')
            run(make_config(input_file, None, stem, xlabel, xlim, [0.0, ymax], labels, pot, channel, detector, loc,
                            variable=variable))
            fig = plt.gcf()
            ax = fig.axes[0]
            fig.canvas.draw()
            box = ax.get_legend().get_window_extent().transformed(ax.transAxes.inverted())
            x0, x1 = ax.transData.inverted().transform(ax.transAxes.transform([[box.x0, 0], [box.x1, 0]]))[:, 0]
            under = inx & (hi > x0) & (lo < x1)
            plt.close('all')
            needed = nice_ceil(top[under].max() / (margin * box.y0)) if under.any() else floor
            if needed <= ymax:
                break
            ymax = needed
        else:
            raise RuntimeError(f'{stem} ch{channel} det{detector} {loc}: ylim did not converge')
        if best is None or ymax < best[0]:
            best = (ymax, loc)
    return [0.0, best[0]], best[1]


TECHNOTE.mkdir(parents=True, exist_ok=True)

for stem, (xlabel, xlim0, xlim1, (detector_labels, pot)) in VARIABLES.items():
    INPUT_FILE = RUNS / f'cov2_{stem}_v1_PROplot_histograms.root'
    variable_token = stem.removesuffix('_sbnd').removesuffix('_icarus')

    for channel, xlim in ((0, xlim0), (1, xlim1)):
        # mutools saves hist_<detector>_<variable>_<code_version>, which carries
        # no channel index; render into a per-channel staging directory, then
        # rename into TECHNOTE.
        outdir = OUTPUT / stem / f'ch{channel}'
        outdir.mkdir(parents=True, exist_ok=True)

        for detector, label in enumerate(detector_labels):
            ylim, legend_loc = padded_ylim(INPUT_FILE, stem, xlabel, xlim, detector_labels, pot, channel, detector)
            run(make_config(INPUT_FILE, outdir, stem, xlabel, xlim, ylim, detector_labels, pot, channel, detector, legend_loc))
            plt.close('all')

            staged = outdir / f'hist_{detector}_0_v3.0.4.pdf'
            target = TECHNOTE / f'datamc_{DETECTOR_TOKENS[label]}_{variable_token}_{CHANNEL_TOKENS[channel]}.pdf'
            shutil.move(staged, target)
            print(f'{target.name:52s} ylim = {ylim}  legend {legend_loc}')


In [ ]:
# ICARUS zoom on the two detector features, from xml/coverage/zoom_icarus.xml:
# the TPC module boundary at z = 0 and the cathode at |x| = 210.12. That config
# holds all four observables in ONE file (the coverage index convention keeps
# var1 = L/E for the model, var2 = True E, var3 = RecoE, var4 = Count), so these
# are selected by variable index rather than by file.
#
# Systematics are off in that XML for now, and PROfit writes the chi2 tree for
# its fitting variable only: var0 carries a chi2, the appended variables do not.
ZOOM_FILE = RUNS / 'cov2_zoom_icarus_v1_PROplot_histograms.root'
ZOOM_LABELS = ['ICARUS Run 2', 'ICARUS Run 4']
ZOOM_POT = ['1.97e19 POT', '1.89e19 POT']
ZOOM_TOKENS = ['icarus_run2', 'icarus_run4']
ZOOM_VARIABLES = {
    # variable: (stem, xlabel, xlim)
    0: ('vertexz_zoom',   'Vertex Z [cm]',     [-50.0, 50.0]),
    5: ('muon_endz_zoom', 'Muon End Z [cm]',   [-50.0, 50.0]),
    6: ('vertexx_zoom',   '|Vertex X| [cm]',   [160.12, 260.12]),
    7: ('muon_endx_zoom', '|Muon End X| [cm]', [160.12, 260.12]),
}

for variable, (stem, xlabel, xlim) in ZOOM_VARIABLES.items():
    for channel in (0, 1):
        outdir = OUTPUT / stem / f'ch{channel}'
        outdir.mkdir(parents=True, exist_ok=True)
        for detector, label in enumerate(ZOOM_LABELS):
            ylim, legend_loc = padded_ylim(ZOOM_FILE, stem, xlabel, xlim, ZOOM_LABELS, ZOOM_POT,
                                           channel, detector, variable=variable)
            run(make_config(ZOOM_FILE, outdir, stem, xlabel, xlim, ylim, ZOOM_LABELS, ZOOM_POT,
                            channel, detector, legend_loc, variable=variable))
            plt.close('all')

            staged = outdir / f'hist_{detector}_{variable}_v3.0.4.pdf'
            target = TECHNOTE / f'datamc_{ZOOM_TOKENS[detector]}_{stem}_{CHANNEL_TOKENS[channel]}.pdf'
            shutil.move(staged, target)
            print(f'{target.name:52s} ylim = {ylim}  legend {legend_loc}')
